# Analytical Darcy MHM with full source moments

The exact harmonic lifts and the separate local source problem implement equation (42) of Harder, Paredes and Valentin (2013). The classical RT0 comparison has different coarse means for this variable source. See the analytical Darcy case page for the complete conventions.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/5ce5bfcfb768c4e3d9ce3a6b0142bbd6afaac445be41642b5d2e83eae002541e/43_analytical_darcy-companion.zip"
COMPANION_SHA256 = "5ce5bfcfb768c4e3d9ce3a6b0142bbd6afaac445be41642b5d2e83eae002541e"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/43_analytical_darcy.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


## Declare and execute the analytical formulation

The live calculation uses the cosine pressure and homogeneous normal data from
the source study on a small mesh. The harmonic energy, face moments and separate
nodal source response are declared in the importable provider below. Assembly,
physical mean fixing and reconstruction use public numerical operations. This
small resolution demonstrates the same equations; the full refinement study
is available through the notebook reproduction task.


In [ ]:
import inspect
import numpy as np
from pymhm import TriangleMesh, assemble
from examples.formulations.analytic_darcy import (
    analytic_equations, define_analytic_darcy,
    analytic_constraints, recover_analytic_darcy,
)

print(inspect.getsource(analytic_equations))

mesh = TriangleMesh.unit_square(2)
def exact_pressure(points):
    return np.prod(np.cos(2 * np.pi * points), axis=-1)
def exact_flux(points):
    x, y = (2 * np.pi * points).T
    return 2 * np.pi * np.column_stack((np.sin(x) * np.cos(y), np.cos(x) * np.sin(y)))
def manufactured_source(points):
    # Independent differentiation: -Delta(p) = 8*pi**2*p.
    return 8 * np.pi**2 * exact_pressure(points)

definition = define_analytic_darcy(
    mesh, source=manufactured_source,
    neumann={int(face): 0.0 for face in mesh.boundary_faces},
    source_degree=2, source_refinement=2, quadrature_order=10,
)
system = assemble(definition.problem)
coefficients = system.solve(constraints=analytic_constraints(definition, system))
solution = recover_analytic_darcy(definition, system, coefficients)
errors = solution.errors(exact_pressure, exact_flux, order=12)
print({"pressure_L2": errors[0], "physical_flux_L2": errors[1],
       "equation_residual": coefficients.residual})
assert np.isfinite(errors).all()
assert coefficients.residual < 1e-10


In [ ]:
import os
if os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1":
    from pathlib import Path
    import json
    from IPython.display import Image, display
    record = json.loads((root / "examples/results/analytic.json").read_text())
    record["results"]

In [ ]:
import os
if os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1":
    rows = record["results"]
    assert len(rows) == 6
    assert max(r["residual"] for r in rows) < 1e-12
    assert rows[-1]["full_pressure_l2"] < rows[0]["full_pressure_l2"]
    assert rows[-1]["full_flux_l2"] < rows[0]["full_flux_l2"]
    assert rows[0]["projected_coarse_l2"] > rows[0]["classical_projected_coarse_l2"]
    display(Image(filename=str(root / "docs/figures/analytic/source-comparison.png")))

The RT0 flux agreement does not imply agreement of the coarse pressure or of the full flux including the source response. Source refinement and quadrature sensitivity are reported independently.

In [ ]:
import os
if os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1":
    record["source_refinement_check"], record["quadrature_check"]